# Threshold Analysis for Token-Level Self-Repair (Google Colab)

This notebook performs comprehensive threshold analysis to understand how different uncertainty thresholds affect:
- Accuracy before and after repair
- Repair trigger rates and success rates
- Uncertainty calibration
- Performance overhead
- Per-dataset performance



In [ ]:
# Step 1: Install dependencies
%pip install -q torch transformers accelerate bitsandbytes huggingface_hub
%pip install -q numpy pandas matplotlib seaborn tqdm scipy rich langgraph langchain-core

# Fix pydantic compatibility (langgraph 1.0.4 works best with pydantic 2.10.x or 2.11.x)
import subprocess
import sys
try:
    import pydantic
    if pydantic.__version__.startswith("2.12"):
        print("⚠️  Pydantic 2.12.x detected. Installing compatible version...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydantic>=2.10.0,<2.12.0", "--force-reinstall"])
        print("✅ Pydantic downgraded for compatibility")
        # Reload pydantic
        import importlib
        import pydantic
        importlib.reload(pydantic)
except Exception as e:
    print(f"Note: {e}")


GITHUB_REPO = "https://github.com/MarthalaSaiKavya/Agentic_LLM"


import sys
from pathlib import Path
import os

# Set up paths for Colab
if 'google.colab' in str(get_ipython()):
    # Running in Colab - Clone from GitHub
    print("Cloning repository from GitHub...")
    
    # Check if already cloned
    PROJECT_PATH = Path('/content/Agentic_LLM')
    
    if not PROJECT_PATH.exists():
        if GITHUB_REPO == "YOUR_USERNAME/Agentic_LLM":
            print("⚠️  Please set GITHUB_REPO variable above to your repository!")
            print("   Example: GITHUB_REPO = 'yourusername/Agentic_LLM'")
            print("   Or use full URL: GITHUB_REPO = 'https://github.com/yourusername/Agentic_LLM.git'")
            raise ValueError("GITHUB_REPO not configured")
        
        # Clone the repository
        if GITHUB_REPO.startswith('http'):
            repo_url = GITHUB_REPO
        else:
            repo_url = f"https://github.com/{GITHUB_REPO}.git"
        
        import subprocess
        result = subprocess.run(['git', 'clone', repo_url, '/content/Agentic_LLM'], 
                              capture_output=True, text=True)
        if result.returncode != 0:
            print(f"❌ Git clone failed: {result.stderr}")
            raise RuntimeError(f"Failed to clone repository: {result.stderr}")
        
        print(f"✅ Repository cloned to: {PROJECT_PATH}")
    else:
        print(f"✅ Repository already exists at: {PROJECT_PATH}")
        print("   (To re-clone, delete the folder first)")
    
    # Change to project directory
    os.chdir(PROJECT_PATH)
    sys.path.insert(0, str(PROJECT_PATH))
    print(f"✅ Working directory: {PROJECT_PATH}")
    
else:
    # Running locally
    PROJECT_PATH = Path().resolve().parent
    sys.path.insert(0, str(PROJECT_PATH))
    print(f"✅ Running locally from: {PROJECT_PATH}")

# Verify imports work
try:
    from src.token_self_repair.llm import load_llama
    from src.token_self_repair.pipelines import default_reasoning_coordinator
    from src.token_self_repair.evaluation import ReasoningEvaluationRunner
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Make sure the repository was cloned correctly and contains the 'src' folder.")


In [ ]:
# Check GPU availability and set quantization accordingly
import torch

if 'google.colab' in str(get_ipython()):
    if torch.cuda.is_available():
        print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
        print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        USE_QUANTIZATION = True  # Use quantization in Colab to save memory
    else:
        print("⚠️  No GPU detected. Using CPU (will be slow).")
        USE_QUANTIZATION = False
else:
    # Local environment - check GPU
    if torch.cuda.is_available():
        print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
        USE_QUANTIZATION = True
    else:
        print("⚠️  No GPU detected. Using CPU.")
        USE_QUANTIZATION = True  # Still use quantization to save memory

import json
import time
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.token_self_repair.llm import load_llama
from src.token_self_repair.pipelines import default_reasoning_coordinator
from src.token_self_repair.evaluation import ReasoningEvaluationRunner
from src.token_self_repair.config import ProjectConfig, Thresholds
from src.token_self_repair.evaluation.reasoning_runner import ReasoningBenchmarkResult

# Set style
plt.style.use('default')  # Use 'default' instead of 'seaborn-v0_8' for Colab compatibility
sns.set_palette("husl")

# Configuration
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
THRESHOLD_VALUES = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
DATASETS = ["gsm8k", "humaneval", "truthfulqa", "bioasq"]  # Add more as needed
MAX_SAMPLES_PER_DATASET = 20  # Adjust based on your needs
RESULTS_DIR = Path("results/threshold_analysis")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nModel: {MODEL_NAME}")
print(f"Quantization: {USE_QUANTIZATION}")
print(f"Thresholds to test: {THRESHOLD_VALUES}")
print(f"Datasets: {DATASETS}")
print(f"Results will be saved to: {RESULTS_DIR}")


## Authenticate with Hugging Face


In [ ]:
# Login to Hugging Face (required for Llama models)
from huggingface_hub import login

# Option 1: Use your token directly (replace with your token)
# login(token="your_hf_token_here")

# Option 2: Use interactive login
print("Please login to Hugging Face:")
login()

print("✅ Hugging Face authentication complete!")


## Load Model


In [ ]:
print("Loading model...")
print("This may take a few minutes on first run (downloading ~16GB)...")
llm = load_llama(model_name=MODEL_NAME, quantize=USE_QUANTIZATION)
print("✅ Model loaded successfully!")


## Data Structures for Results


In [ ]:
@dataclass
class ThresholdResult:
    """Results for a single threshold value."""
    threshold: float
    dataset: str
    accuracy_before: float  # Accuracy without repair
    accuracy_after: float    # Accuracy with repair
    accuracy_improvement: float
    repair_trigger_rate: float  # % of queries that triggered repair
    avg_repairs_per_query: float
    repair_success_rate: float  # % of repairs that improved output
    false_positive_rate: float  # Repairs triggered but not needed
    false_negative_rate: float   # Needed repairs not triggered
    auroc: float
    calibration_error: float
    avg_uncertainty: float
    avg_latency: float
    latency_overhead: float  # Additional time due to repair
    num_samples: int
    timestamp: str

@dataclass
class RepairCase:
    """Individual repair case study."""
    threshold: float
    dataset: str
    prompt: str
    reference: str
    prediction_before: str
    prediction_after: str
    uncertainty_before: float
    uncertainty_after: float
    correct_before: bool
    correct_after: bool
    repair_triggered: bool
    num_repairs: int
    success: bool  # Did repair improve correctness

# Storage
all_results: List[ThresholdResult] = []
case_studies: List[RepairCase] = []


## Visualization: Uncertainty Calibration & Performance


In [ ]:
if len(df_results) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Uncertainty Calibration & Performance Analysis', fontsize=16, fontweight='bold')

    # Plot 1: AUROC vs Threshold
    ax1 = axes[0, 0]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax1.plot(subset['threshold'], subset['auroc'], 'o-', label=dataset, linewidth=2)
    ax1.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random')
    ax1.set_xlabel('Threshold')
    ax1.set_ylabel('AUROC')
    ax1.set_title('AUROC vs Threshold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot 2: Calibration Error vs Threshold
    ax2 = axes[0, 1]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax2.plot(subset['threshold'], subset['calibration_error'], 'o-', label=dataset, linewidth=2)
    ax2.set_xlabel('Threshold')
    ax2.set_ylabel('Expected Calibration Error')
    ax2.set_title('Calibration Error vs Threshold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Plot 3: Average Uncertainty vs Threshold
    ax3 = axes[0, 2]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax3.plot(subset['threshold'], subset['avg_uncertainty'], 'o-', label=dataset, linewidth=2)
    ax3.set_xlabel('Threshold')
    ax3.set_ylabel('Average Uncertainty')
    ax3.set_title('Average Uncertainty vs Threshold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # Plot 4: Latency Overhead
    ax4 = axes[1, 0]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax4.plot(subset['threshold'], subset['latency_overhead'], 'o-', label=dataset, linewidth=2)
    ax4.set_xlabel('Threshold')
    ax4.set_ylabel('Latency Overhead (seconds)')
    ax4.set_title('Repair Latency Overhead vs Threshold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    # Plot 5: False Positive Rate
    ax5 = axes[1, 1]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax5.plot(subset['threshold'], subset['false_positive_rate'] * 100, 'o-', label=dataset, linewidth=2)
    ax5.set_xlabel('Threshold')
    ax5.set_ylabel('False Positive Rate (%)')
    ax5.set_title('False Positive Rate (Unnecessary Repairs)')
    ax5.legend()
    ax5.grid(True, alpha=0.3)

    # Plot 6: False Negative Rate
    ax6 = axes[1, 2]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax6.plot(subset['threshold'], subset['false_negative_rate'] * 100, 'o-', label=dataset, linewidth=2)
    ax6.set_xlabel('Threshold')
    ax6.set_ylabel('False Negative Rate (%)')
    ax6.set_title('False Negative Rate (Missed Repairs)')
    ax6.legend()
    ax6.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = RESULTS_DIR / f"calibration_performance_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"Plot saved to: {plot_path}")
    
    # Download plot from Colab
    if 'google.colab' in str(get_ipython()):
        from google.colab import files
        files.download(str(plot_path))
    
    plt.show()
else:
    print("No results to plot. Run the evaluation loop first.")


## Optimal Threshold Analysis


In [ ]:
if len(df_results) > 0:
    print("=== Optimal Threshold Analysis ===\n")
    
    optimal_thresholds = {}
    
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset]
        
        # Find threshold with maximum accuracy improvement
        best_idx = subset['accuracy_improvement'].idxmax()
        best_row = subset.loc[best_idx]
        
        optimal_thresholds[dataset] = {
            'threshold': best_row['threshold'],
            'accuracy_improvement': best_row['accuracy_improvement'],
            'accuracy_after': best_row['accuracy_after'],
            'repair_trigger_rate': best_row['repair_trigger_rate'],
            'repair_success_rate': best_row['repair_success_rate'],
        }
        
        print(f"Dataset: {dataset}")
        print(f"  Optimal Threshold: {best_row['threshold']:.3f}")
        print(f"  Accuracy Improvement: {best_row['accuracy_improvement']:+.3f}")
        print(f"  Final Accuracy: {best_row['accuracy_after']:.3f}")
        print(f"  Repair Trigger Rate: {best_row['repair_trigger_rate']*100:.1f}%")
        print(f"  Repair Success Rate: {best_row['repair_success_rate']*100:.1f}%")
        print()
    
    # Overall optimal (average across datasets)
    avg_improvements = df_results.groupby('threshold')['accuracy_improvement'].mean()
    overall_optimal = avg_improvements.idxmax()
    print(f"Overall Optimal Threshold (avg across datasets): {overall_optimal:.3f}")
    print(f"Average Accuracy Improvement: {avg_improvements[overall_optimal]:+.3f}")
else:
    print("No results available. Run the evaluation loop first.")


## Summary Table


In [ ]:
if len(df_results) > 0:
    # Create summary table
    summary_data = []
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset]
        
        best_idx = subset['accuracy_improvement'].idxmax()
        best = subset.loc[best_idx]
        
        summary_data.append({
            'Dataset': dataset,
            'Optimal Threshold': f"{best['threshold']:.3f}",
            'Baseline Accuracy': f"{best['accuracy_before']:.3f}",
            'Best Accuracy': f"{best['accuracy_after']:.3f}",
            'Improvement': f"{best['accuracy_improvement']:+.3f}",
            'Repair Trigger %': f"{best['repair_trigger_rate']*100:.1f}",
            'Repair Success %': f"{best['repair_success_rate']*100:.1f}",
            'AUROC': f"{best['auroc']:.3f}",
            'Calibration Error': f"{best['calibration_error']:.3f}",
            'Latency Overhead (s)': f"{best['latency_overhead']:.2f}",
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n=== Summary Table ===")
    print(summary_df.to_string(index=False))
    
    # Save summary
    summary_path = RESULTS_DIR / f"summary_table_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSummary saved to: {summary_path}")
    
    # Download from Colab
    if 'google.colab' in str(get_ipython()):
        from google.colab import files
        files.download(str(summary_path))
else:
    print("No results available. Run the evaluation loop first.")


## Final Summary and Recommendations


In [ ]:
if len(df_results) > 0:
    print("\n" + "="*80)
    print("FINAL SUMMARY AND RECOMMENDATIONS")
    print("="*80)
    
    # Calculate overall optimal
    avg_improvements = df_results.groupby('threshold')['accuracy_improvement'].mean()
    overall_optimal = avg_improvements.idxmax()
    
    print("\n1. OPTIMAL THRESHOLDS BY DATASET:")
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset]
        best_idx = subset['accuracy_improvement'].idxmax()
        best_row = subset.loc[best_idx]
        print(f"   {dataset:15s}: {best_row['threshold']:.3f} (improvement: {best_row['accuracy_improvement']:+.3f})")
    
    print(f"\n2. OVERALL OPTIMAL THRESHOLD: {overall_optimal:.3f}")
    
    print("\n3. KEY INSIGHTS:")
    avg_auroc = df_results['auroc'].mean()
    avg_cal_error = df_results['calibration_error'].mean()
    avg_overhead = df_results['latency_overhead'].mean()
    
    print(f"   - Average AUROC: {avg_auroc:.3f}")
    print(f"   - Average Calibration Error: {avg_cal_error:.3f}")
    print(f"   - Average Latency Overhead: {avg_overhead:.2f}s")
    
    print("\n4. RECOMMENDATIONS:")
    print(f"   - Use threshold {overall_optimal:.3f} for general-purpose use")
    print("   - Consider dataset-specific thresholds for specialized applications")
    print("   - Monitor repair trigger rates to balance accuracy vs. performance")
    print("   - Review case studies to understand repair behavior")
    
    print("\n" + "="*80)
else:
    print("No results available. Run the evaluation loop first.")


## Helper Functions


In [ ]:
def run_baseline_evaluation(dataset_name: str, max_samples: int) -> ReasoningBenchmarkResult:
    """Run evaluation without repair (baseline)."""
    def factory():
        coordinator = default_reasoning_coordinator(llm)
        coordinator.config.thresholds.repair_activation_uncertainty = 1.0
        coordinator.config.max_self_repairs = 0
        return coordinator
    runner = ReasoningEvaluationRunner(coordinator_factory=factory)
    return runner.run(dataset_name, max_samples=max_samples)

def run_threshold_evaluation(dataset_name: str, threshold: float, max_samples: int) -> ReasoningBenchmarkResult:
    """Run evaluation with a specific threshold."""
    def factory():
        coordinator = default_reasoning_coordinator(llm)
        coordinator.config.thresholds.repair_activation_uncertainty = threshold
        coordinator.config.max_self_repairs = 3
        return coordinator
    runner = ReasoningEvaluationRunner(coordinator_factory=factory)
    return runner.run(dataset_name, max_samples=max_samples)

def calculate_repair_metrics(baseline_result: ReasoningBenchmarkResult, 
                            repair_result: ReasoningBenchmarkResult) -> Dict:
    """Calculate repair-specific metrics."""
    repairs_triggered = sum(1 for s in repair_result.samples if s.final_uncertainty > 0.3)
    total_repairs = repairs_triggered
    successful_repairs = 0
    false_positives = 0
    false_negatives = 0
    
    baseline_dict = {s.prompt: s for s in baseline_result.samples}
    for repair_sample in repair_result.samples:
        baseline_sample = baseline_dict.get(repair_sample.prompt)
        if not baseline_sample:
            continue
        triggered = repair_sample.final_uncertainty > 0.3
        if triggered:
            if not baseline_sample.correct and repair_sample.correct:
                successful_repairs += 1
            elif baseline_sample.correct and not repair_sample.correct:
                false_positives += 1
        else:
            if not baseline_sample.correct and not repair_sample.correct:
                false_negatives += 1
    
    num_samples = len(repair_result.samples)
    return {
        "repair_trigger_rate": repairs_triggered / num_samples if num_samples > 0 else 0.0,
        "avg_repairs_per_query": total_repairs / num_samples if num_samples > 0 else 0.0,
        "repair_success_rate": successful_repairs / repairs_triggered if repairs_triggered > 0 else 0.0,
        "false_positive_rate": false_positives / repairs_triggered if repairs_triggered > 0 else 0.0,
        "false_negative_rate": false_negatives / (num_samples - repairs_triggered) if (num_samples - repairs_triggered) > 0 else 0.0,
    }


In [ ]:
print("Starting threshold sweep evaluation...")
print(f"This will test {len(THRESHOLD_VALUES)} thresholds across {len(DATASETS)} datasets")
print(f"Total evaluations: {len(THRESHOLD_VALUES) * len(DATASETS) * 2} (baseline + repair for each)")
print("\nThis may take a while. Progress will be shown below.\n")

baseline_results = {}

# First, run baseline evaluations (no repair)
print("\n=== Running Baseline Evaluations (No Repair) ===")
for dataset in DATASETS:
    print(f"\nDataset: {dataset}")
    try:
        baseline = run_baseline_evaluation(dataset, MAX_SAMPLES_PER_DATASET)
        baseline_results[dataset] = baseline
        print(f"  Baseline Accuracy: {baseline.accuracy:.3f}")
    except Exception as e:
        print(f"  ❌ Error: {e}")
        baseline_results[dataset] = None

# Now run threshold sweep
print("\n\n=== Running Threshold Sweep ===")
for threshold in tqdm(THRESHOLD_VALUES, desc="Thresholds"):
    for dataset in DATASETS:
        if baseline_results.get(dataset) is None:
            continue
        try:
            repair_result = run_threshold_evaluation(dataset, threshold, MAX_SAMPLES_PER_DATASET)
            baseline = baseline_results[dataset]
            repair_metrics = calculate_repair_metrics(baseline, repair_result)
            latency_overhead = repair_result.average_latency - baseline.average_latency
            
            result = ThresholdResult(
                threshold=threshold, dataset=dataset,
                accuracy_before=baseline.accuracy, accuracy_after=repair_result.accuracy,
                accuracy_improvement=repair_result.accuracy - baseline.accuracy,
                repair_trigger_rate=repair_metrics["repair_trigger_rate"],
                avg_repairs_per_query=repair_metrics["avg_repairs_per_query"],
                repair_success_rate=repair_metrics["repair_success_rate"],
                false_positive_rate=repair_metrics["false_positive_rate"],
                false_negative_rate=repair_metrics["false_negative_rate"],
                auroc=repair_result.auroc, calibration_error=repair_result.calibration_error,
                avg_uncertainty=repair_result.average_uncertainty,
                avg_latency=repair_result.average_latency, latency_overhead=latency_overhead,
                num_samples=len(repair_result.samples), timestamp=datetime.now().isoformat(),
            )
            all_results.append(result)
        except Exception as e:
            print(f"\n❌ Error at threshold={threshold}, dataset={dataset}: {e}")
            continue

print("\n✅ Evaluation complete!")


In [ ]:
# Convert to DataFrame for easier analysis
df_results = pd.DataFrame([asdict(r) for r in all_results])

# Save to CSV
csv_path = RESULTS_DIR / f"threshold_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_results.to_csv(csv_path, index=False)
print(f"Results saved to: {csv_path}")

# Save raw JSON
json_path = RESULTS_DIR / f"threshold_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(json_path, 'w') as f:
    json.dump([asdict(r) for r in all_results], f, indent=2)
print(f"Raw results saved to: {json_path}")

# Download from Colab (if running in Colab)
if 'google.colab' in str(get_ipython()):
    from google.colab import files
    files.download(str(csv_path))
    files.download(str(json_path))
    print("✅ Files downloaded to your computer!")

# Display summary
print("\n=== Summary ===")
print(f"Total evaluations: {len(all_results)}")
if len(df_results) > 0:
    print(f"Datasets: {df_results['dataset'].unique()}")
    print(f"Thresholds tested: {sorted(df_results['threshold'].unique())}")


## Visualization: Accuracy vs Threshold


In [ ]:
if len(df_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Threshold Analysis: Accuracy Metrics', fontsize=16, fontweight='bold')

    # Plot 1: Accuracy Before vs After
    ax1 = axes[0, 0]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax1.plot(subset['threshold'], subset['accuracy_before'], 'o--', label=f'{dataset} (before)', alpha=0.7)
        ax1.plot(subset['threshold'], subset['accuracy_after'], 'o-', label=f'{dataset} (after)', linewidth=2)
    ax1.set_xlabel('Threshold')
    ax1.set_ylabel('Accuracy')
    ax1.set_title('Accuracy Before vs After Repair')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot 2: Accuracy Improvement
    ax2 = axes[0, 1]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax2.plot(subset['threshold'], subset['accuracy_improvement'], 'o-', label=dataset, linewidth=2)
    ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Threshold')
    ax2.set_ylabel('Accuracy Improvement')
    ax2.set_title('Accuracy Improvement by Threshold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Plot 3: Repair Trigger Rate
    ax3 = axes[1, 0]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax3.plot(subset['threshold'], subset['repair_trigger_rate'] * 100, 'o-', label=dataset, linewidth=2)
    ax3.set_xlabel('Threshold')
    ax3.set_ylabel('Repair Trigger Rate (%)')
    ax3.set_title('Repair Trigger Rate vs Threshold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # Plot 4: Repair Success Rate
    ax4 = axes[1, 1]
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        ax4.plot(subset['threshold'], subset['repair_success_rate'] * 100, 'o-', label=dataset, linewidth=2)
    ax4.set_xlabel('Threshold')
    ax4.set_ylabel('Repair Success Rate (%)')
    ax4.set_title('Repair Success Rate vs Threshold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = RESULTS_DIR / f"accuracy_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"Plot saved to: {plot_path}")
    
    # Download plot from Colab
    if 'google.colab' in str(get_ipython()):
        from google.colab import files
        files.download(str(plot_path))
    
    plt.show()
else:
    print("No results to plot. Run the evaluation loop first.")
